# Exploration of Vermivora data in GBIF


## 0. Install needed packages

`rioxarray` is a Python library built on [GDAL](https://gdal.org/en/stable/) for reading, writing, and analyzing 
gridded raster data like GeoTIFFs, satellite images, and digital elevation models

In [3]:
pip install rioxarray
pip install pygbif

SyntaxError: invalid syntax (2109096242.py, line 1)

## 1. Import packages

We will use:

* `geopandas` for the Maine boundary and occurrence points
* `rioxarray` for opening and clipping rasters
* `xarray` for selecting raster values at many points
* `pathlib` is useful for manipulating filesystem paths

In [2]:
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
from pathlib import Path

from pygbif import species
from pygbif import occurrences as occ

ModuleNotFoundError: No module named 'pygbif'

## 4. Open annual mean temperature

`rxr.open_rasterio()` opens a GeoTIFF as an `xarray.DataArray`. The option `masked=True` converts the raster's NoData values to missing values.

At this point we are only opening the raster. We have not clipped it or extracted any values.

In [ ]:
# Create the Path to the bioclim data directory, then below you can
# join the directory to a specific file in a straightforward way
bioclim_path = Path("/home/jovyan/shared/bioclim")

bio1 = rxr.open_rasterio(
    bioclim_path / "wc2.1_2.5m_bio_1.tif",
    masked=True,
)

bio1

In [ ]:
bio1.squeeze().plot() # squeeze flattens the one band into a usable axis

## 5. Inspect raster dimensions

The raster has one band, 4,320 rows, and 8,640 columns. The `x` and `y` coordinates identify cell locations.

In [ ]:
print("Dimensions:", bio1.dims)
print("Shape:", bio1.shape) # number of dimensions
print("Data type:", bio1.dtype)

## 6. Inspect CRS, resolution, extent, and NoData

These properties determine where the raster is located and how its cells line up.

The resolution is expressed in degrees because this raster uses longitude and latitude. A 2.5 arc-minute cell is 0.04167 degree tall and wide. Its physical width changes with latitude.

In [6]:
print("CRS:", bio1.rio.crs)
print("Resolution:", bio1.rio.resolution()) # size of grid cells in crs unit (decimal degrees)
print("Bounds:", bio1.rio.bounds())
print("NoData:", bio1.rio.nodata)

CRS: EPSG:4326
Resolution: (0.041666666666666664, -0.041666666666666664)
Bounds: (-180.0, -89.99999999999997, 180.0, 90.0)
NoData: nan


## 10. Open annual precipitation

BIO12 represents annual precipitation in millimeters. Open it before checking whether it aligns with BIO1.

In [7]:
bio12 = rxr.open_rasterio(
    bioclim_path / "wc2.1_2.5m_bio_12.tif",
    masked=True,
)

## 11. Check raster alignment

Two rasters are aligned when their CRS, dimensions, resolution, and cell locations agree. Alignment matters because the same row and column should refer to the same place.

These WorldClim files were created as one aligned set.

In [8]:
print("Same CRS:", bio1.rio.crs == bio12.rio.crs)
print("Same shape:", bio1.shape == bio12.shape)
print("Same resolution:", bio1.rio.resolution() == bio12.rio.resolution())
print("Same bounds:", bio1.rio.bounds() == bio12.rio.bounds())

Same CRS: True
Same shape: True
Same resolution: True
Same bounds: True


## 15. Confirm the clipped rasters are aligned

The four clipped rasters should have the same shape and coordinates. This allows values from the same cell locations to be compared.

## 16. Load the Vermivora occurrence records

Search for name suggestions.

In [ ]:
gw_suggestions = species.name_suggest(q="Vermivora chrysoptera")
# gw_suggestions will have a list of dictionaries
# select the first element of this list here and save it as a new variable called `gw_match`

gw_match = gw_suggestions[1]
gw_match

In [42]:
## Remove rows without species labels
amphibians_gdf = amphibians_gdf[amphibians_gdf["species"] != ""]
amphibians_gdf.shape

(25586, 51)

## 17. Select only the relevant columns
Just to reduce the amount of clutter we'll subset the columns to keep only the few
that are relevant (notice that we are also explicitly keeping the 'geometry' column).

In [43]:
columns_to_keep = [
    "gbifID",
    "species",
    "decimalLongitude",
    "decimalLatitude",
    "year",
    "geometry",
]

amphibians_gdf = amphibians_gdf[columns_to_keep]

amphibians_gdf.head()

,gbifID,species,decimalLongitude,decimalLatitude,year,geometry
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,2026,POINT (-68.66028 44.90255)
1,6520735611,Aquarana clamitans,-69.649014,44.735713,2026,POINT (-69.64901 44.73571)
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,2026,POINT (-69.88924 44.79832)
3,6520721493,Lithobates palustris,-68.636788,44.860466,2022,POINT (-68.63679 44.86047)
4,6520718655,Lithobates pipiens,-68.788851,44.990948,2026,POINT (-68.78885 44.99095)


## 18. Verify agreement in CRS for occurences and bio1

The occurrence coordinates and climate rasters both use `EPSG:4326`. Matching coordinate systems are required before extracting raster values.

In [44]:
print(amphibians_gdf.crs)
print(bio1_maine.rio.crs)

EPSG:4326
EPSG:4326


## 19. Prepare point coordinates for raster extraction

`xarray` can select values for many coordinates at once. Create one array of occurrence longitudes and one array of occurrence latitudes.

In [45]:
point_x = xr.DataArray(
    amphibians_gdf.geometry.x.to_numpy(),
    dims="record",
)

point_y = xr.DataArray(
    amphibians_gdf.geometry.y.to_numpy(),
    dims="record",
)

## 20. Extract annual mean temperature

`sel()` selects raster cells using their coordinates. `method="nearest"` selects the center of the cell nearest each occurrence coordinate.

Every occurrence assigned to the same raster cell receives the same climate value.

In [46]:
bio1_values = bio1_maine.sel(
    x=point_x,
    y=point_y,
    method="nearest",
)

amphibians_gdf["bio1_mean_temp_c"] = bio1_values
amphibians_gdf.head()

,gbifID,species,decimalLongitude,decimalLatitude,year,geometry,bio1_mean_temp_c
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,2026,POINT (-68.66028 44.90255),6.609500
1,6520735611,Aquarana clamitans,-69.649014,44.735713,2026,POINT (-69.64901 44.73571),6.332833
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,2026,POINT (-69.88924 44.79832),5.896000
3,6520721493,Lithobates palustris,-68.636788,44.860466,2022,POINT (-68.63679 44.86047),6.544000
4,6520718655,Lithobates pipiens,-68.788851,44.990948,2026,POINT (-68.78885 44.99095),6.283834


## 21. Check missing extracted values

An extracted value can be missing when a point is nearest to a NoData cell. This is common near coastlines because raster cells are much larger and less detailed than the state boundary.

Do not automatically replace missing climate values with zero. Zero degrees or zero precipitation would be real environmental values, not missing-data codes.

In [38]:
amphibians_gdf["bio1_mean_temp_c"].isna().value_counts()

bio1_mean_temp_c
False    23654
True      1932
Name: count, dtype: int64

For the sake of simplicity we will just remove any cells without environmental data

In [39]:
# amphibians_gdf = amphibians_gdf.dropna() # skip for now because throws off alignment with point_x and point_y

## 22. Extract the other climate variables

Use the same point coordinates for each aligned raster.

In [47]:
bio5_values = bio5_maine.sel(x=point_x, y=point_y, method="nearest")
amphibians_gdf["bio5_warmest_max_c"] = bio5_values

In [48]:
bio6_values = bio6_maine.sel(x=point_x, y=point_y, method="nearest")
amphibians_gdf["bio6_coldest_min_c"] = bio6_values

In [52]:
bio12_values = bio12_maine.sel(x=point_x, y=point_y, method="nearest")
amphibians_gdf["bio12_annual_precip_mm"] = bio12_values
amphibians_gdf.head()

,gbifID,species,decimalLongitude,decimalLatitude,year,geometry,bio1_mean_temp_c,bio5_warmest_max_c,bio6_coldest_min_c,bio12_annual_precip_mm
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,2026,POINT (-68.66028 44.90255),6.609500,26.820,-13.808000,1044.0
1,6520735611,Aquarana clamitans,-69.649014,44.735713,2026,POINT (-69.64901 44.73571),6.332833,26.612,-14.944000,1051.0
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,2026,POINT (-69.88924 44.79832),5.896000,26.236,-15.900000,1090.0
3,6520721493,Lithobates palustris,-68.636788,44.860466,2022,POINT (-68.63679 44.86047),6.544000,26.608,-13.752000,1057.0
4,6520718655,Lithobates pipiens,-68.788851,44.990948,2026,POINT (-68.78885 44.99095),6.283834,26.424,-14.415999,1043.0


In [50]:
amphibians_gdf.explore()

## 23. Create the occurrence-environment table

The geometry column is useful for spatial analysis but is not needed in an ordinary analysis table. Drop it while keeping species, coordinates, year, and environmental covariates.

In [55]:
occurrence_environment = amphibians_gdf.drop(columns="geometry")

occurrence_environment.head()

,gbifID,species,decimalLongitude,decimalLatitude,year,bio1_mean_temp_c,bio5_warmest_max_c,bio6_coldest_min_c,bio12_annual_precip_mm
0,6520738729,Lithobates catesbeianus,-68.660278,44.902551,2026,6.609500,26.820,-13.808000,1044.0
1,6520735611,Aquarana clamitans,-69.649014,44.735713,2026,6.332833,26.612,-14.944000,1051.0
2,6520729466,Boreorana sylvatica,-69.889241,44.798325,2026,5.896000,26.236,-15.900000,1090.0
3,6520721493,Lithobates palustris,-68.636788,44.860466,2022,6.544000,26.608,-13.752000,1057.0
4,6520718655,Lithobates pipiens,-68.788851,44.990948,2026,6.283834,26.424,-14.415999,1043.0


## 24. Summarize climate values by species

These are summaries of the environments represented by occurrence records. They are not unbiased estimates of a species' full climatic niche because sampling locations are uneven.

In [60]:
predictor_columns = [
    "bio1_mean_temp_c",
    "bio5_warmest_max_c",
    "bio6_coldest_min_c",
    "bio12_annual_precip_mm",
]

for species, samples in occurrence_environment.groupby("species"):
    print("Mean: ", species, len(samples), "\n", samples[predictor_columns].mean())
#    print("SD: ", species, len(samples), "\n", samples[predictor_columns].std())


Mean:  Ambystoma jeffersonianum 37 
 bio1_mean_temp_c             4.399309
bio5_warmest_max_c          25.026588
bio6_coldest_min_c         -17.719530
bio12_annual_precip_mm    1028.352941
dtype: float64
Mean:  Ambystoma laterale 322 
 bio1_mean_temp_c             6.185677
bio5_warmest_max_c          26.245603
bio6_coldest_min_c         -14.443116
bio12_annual_precip_mm    1076.842767
dtype: float64
Mean:  Ambystoma maculatum 2511 
 bio1_mean_temp_c             6.158210
bio5_warmest_max_c          25.551662
bio6_coldest_min_c         -13.933883
bio12_annual_precip_mm    1144.519107
dtype: float64
Mean:  Ambystoma tigrinum 1 
 bio1_mean_temp_c             7.269667
bio5_warmest_max_c          26.708000
bio6_coldest_min_c         -12.692000
bio12_annual_precip_mm    1148.000000
dtype: float64
Mean:  Anaxyrus americanus 2800 
 bio1_mean_temp_c             5.911439
bio5_warmest_max_c          25.408857
bio6_coldest_min_c         -14.351806
bio12_annual_precip_mm    1134.465169
dtype: float6

## 25. Check relationships among predictors

Environmental predictors can contain similar information. A correlation table is a simple first check for redundancy. High correlation does not prove that one variable should be removed, but it should inform predictor selection and interpretation.

In [61]:
occurrence_environment[predictor_columns].corr().round(2)

,bio1_mean_temp_c,bio5_warmest_max_c,bio6_coldest_min_c,bio12_annual_precip_mm
bio1_mean_temp_c,1.00,0.56,0.94,0.41
bio5_warmest_max_c,0.56,1.00,0.27,-0.22
bio6_coldest_min_c,0.94,0.27,1.00,0.57
bio12_annual_precip_mm,0.41,-0.22,0.57,1.00


## 26. Save the analysis table

Saving the table separates the expensive spatial preparation from later statistical analysis. Missing raster values remain missing in the CSV.

In [62]:
occurrence_environment.to_csv(
    "Lab05_environmental_values.csv",
    index=False,
)

## 27. Save the geodataframe
Saving your geodataframe to a file allows you to load it back in at a future time
without needing to redo all the analysis. geojson is a text-based file format based 
on JSON used to encode geographic data structures and non-spatial attributes.

In [63]:
amphibians_gdf.to_file("Lab05_amphibians_gdf.geojson")

## Discussion questions

1. How did clipping change raster extent and shape? What properties stayed the same?
2. Why must rasters and occurrence points use compatible coordinate systems?
3. Why should missing extracted values remain missing instead of being replaced by zero?
4. What would be some alternative strategies for dealing with No Data values when extracting environmental data?
5. Which is more biologically meaningful for an amphibian: annual mean temperature, summer maximum temperature, or winter minimum temperature? Why might the answer differ among species?
6. What important fine-scale environmental variation is missed by a 2.5 arc-minute climate raster?

## Submit your work to GitHub

Run the notebook from top to bottom. Save it, then add, commit, and push the notebook from a terminal.

```bash
git add docs/labs/Lab05-RasterData.ipynb
git commit -m "Complete raster data lab"
git push
```